# FibRec, FibMemo e FibIt (C++)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/FibRec_FibMemo_FibIt_cpp.ipynb)

Capitolo 18 — Algoritmi ricorsivi e iterativi a confronto. Notebook eseguibile di `FibRec`, `FibMemo` e `FibIt`: gli algoritmi in C++, eseguiti sugli esempi del capitolo, con la cella «Prova tu» per provarli con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/18-ricorsivo-iterativo/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/FibRec_FibMemo_FibIt_python.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/FibRec_FibMemo_FibIt_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo_cpp.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Ogni programma C++ è scritto su un file dalla cella `%%writefile`, poi compilato con `g++` ed eseguito nella cella successiva: tutto avviene in Colab.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `FibRec` → `FibRec` (PascalCase).

## Fibonacci: FibRec, FibMemo e FibIt (sottoproblemi che si sovrappongono)

*Input*: un naturale $n$. *Output*: il numero di Fibonacci $F(n)$, con $F(0)=0$, $F(1)=1$, $F(n)=F(n-1)+F(n-2)$.

- **FibRec**$(n)$ — traduzione diretta della definizione: i sottoproblemi si sovrappongono e le chiamate sono $2F(n+1)-1=\Theta(\varphi^n)$, esponenziali; la profondità è $n$.
- **FibMemo**$(n,memo)$ — ricorsione con memoizzazione: ogni valore è calcolato una volta (il vettore `memo` vale $-1$ per «non definito»); $2n-1$ chiamate, memoria $\Theta(n)$.
- **FibIt**$(n)$ — ciclo con due variabili: $n-1$ iterazioni, memoria $\Theta(1)$.

Sull'esempio $\texttt{FibRec}(5)$ del capitolo si conta quante volte è chiamata ogni sottoproblema (l'albero ha $15$ nodi, ma servono solo $6$ valori diversi). Poi la tabella dei conteggi (che coincidono con i valori del capitolo: $21\,891$, $242\,785$, $2\,692\,537$ chiamate per $n=20,25,30$) e i tempi misurati.

In [1]:
%%writefile fibonacci.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

long long fibCalls = 0;   // chiamate eseguite

// FibRec(n)
long long FibRec(int n)
{
    fibCalls++;
    if (n <= 1)
    {
        return n;
    }
    return FibRec(n - 1) + FibRec(n - 2);
}

// FibMemo(n, memo): memo[k] = -1 significa «non definito»
long long FibMemo(int n, long long *memo)
{
    fibCalls++;
    if (n <= 1)
    {
        return n;
    }
    if (memo[n] == -1)
    {
        memo[n] = FibMemo(n - 1, memo) + FibMemo(n - 2, memo);
    }
    return memo[n];
}

// FibIt(n)
long long FibIt(int n)
{
    if (n <= 1)
    {
        return n;
    }
    long long prev = 0;             // F(i-2)
    long long curr = 1;             // F(i-1)
    int i = 2;
    while (i <= n)
    {
        long long next = prev + curr;
        prev = curr;
        curr = next;
        i = i + 1;
    }
    return curr;
}

// Come FibRec, ma conta le chiamate per ogni valore del parametro
long long FibRecHist(int n, int *counts)
{
    counts[n]++;
    if (n <= 1)
    {
        return n;
    }
    return FibRecHist(n - 1, counts) + FibRecHist(n - 2, counts);
}

double NowMs()
{
    return chrono::duration<double, milli>(chrono::steady_clock::now().time_since_epoch()).count();
}

#ifndef NOMAIN
int main()
{
    int counts[6] = {0, 0, 0, 0, 0, 0};
    long long f5 = FibRecHist(5, counts);
    printf("FibRec(5) = %lld\n", f5);
    string line;
    int total = 0;
    for (int k = 5; k >= 0; k--)
    {
        if (k < 5) line += "  ";
        line += to_string(k) + ":" + to_string(counts[k]);
        total += counts[k];
    }
    printf("chiamate per ogni valore del parametro: %s   totale %d\n", line.c_str(), total);
    printf("\n");
    printf("%3s %8s %9s %10s %8s %6s  (chiamate, chiamate, chiamate, iterazioni)\n", "n", "F(n)", "FibRec", "2F(n+1)-1", "FibMemo", "FibIt");
    int ns[] = {10, 15, 20, 25, 30};
    for (int n : ns)
    {
        long long f = FibIt(n);
        fibCalls = 0;
        long long r1 = FibRec(n);
        long long c1 = fibCalls;
        fibCalls = 0;
        vector<long long> memo(n + 1, -1);
        long long r2 = FibMemo(n, memo.data());
        long long c2 = fibCalls;
        bool ok = (r1 == r2 && r2 == f && c1 == 2 * FibIt(n + 1) - 1 && c2 == 2 * n - 1);
        printf("%3d %8lld %9lld %10lld %8lld %6d  %s\n", n, f, c1, 2 * FibIt(n + 1) - 1, c2, n - 1, ok ? "OK" : "ERROR");
    }
    printf("\n");
    printf("Tempi misurati (dipendono dalla macchina):\n");
    for (int n : {20, 25, 30})
    {
        double t0 = NowMs(); volatile long long sink = FibRec(n); double t1 = NowMs() - t0;
        t0 = NowMs();
        for (int rep = 0; rep < 1000; rep++)
        {
            sink = FibIt(n);
        }
        double t2 = (NowMs() - t0) / 1000;
        printf("n=%-3d FibRec %10.4f ms   FibIt %10.6f ms\n", n, t1, t2);
    }
    return 0;
}
#endif

Writing fibonacci.cpp


In [2]:
!g++ -std=c++17 -O2 -o fibonacci fibonacci.cpp && ./fibonacci

FibRec(5) = 5
chiamate per ogni valore del parametro: 5:1  4:1  3:2  2:3  1:5  0:3   totale 15

  n     F(n)    FibRec  2F(n+1)-1  FibMemo  FibIt  (chiamate, chiamate, chiamate, iterazioni)
 10       55       177        177       19      9  OK
 15      610      1973       1973       29     14  OK
 20     6765     21891      21891       39     19  OK
 25    75025    242785     242785       49     24  OK
 30   832040   2692537    2692537       59     29  OK

Tempi misurati (dipendono dalla macchina):
n=20  FibRec     0.0156 ms   FibIt   0.000009 ms
n=25  FibRec     0.2075 ms   FibIt   0.000011 ms
n=30  FibRec     2.3123 ms   FibIt   0.000019 ms


**Prova tu.** Aumenta $n$ (con $n=32$ `FibRec` esegue $7\,049\,155$ chiamate, `FibMemo` $63$) e guarda come cresce il tempo di `FibRec`.

In [3]:
%%writefile prova_fibonacci.cpp
#define NOMAIN
#include "fibonacci.cpp"

int main()
{
    // Aumenta n: con n = 32 FibRec esegue 7 049 155 chiamate, FibMemo 63.
    int n = 32;
    fibCalls = 0;
    long long r1 = FibRec(n);
    printf("%lld %lld\n", r1, fibCalls);
    fibCalls = 0;
    vector<long long> memo(n + 1, -1);
    long long r2 = FibMemo(n, memo.data());
    printf("%lld %lld\n", r2, fibCalls);
    return 0;
}

Writing prova_fibonacci.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_fibonacci prova_fibonacci.cpp && ./prova_fibonacci

2178309 7049155
2178309 63


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).